<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

In [ ]:
%pip install mistralai python-dotenv

In [14]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

load_dotenv()

if "MISTRAL_API_KEY" not in os.environ:
    raise ValueError("MISTRAL_API_KEY not found in environment variables")

mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  2. Working with Different Modalities
</h2>

**Objectives**: Expand knowledge gained in the previous notebook to work with additional modalities, not just text

**Difficulty**: 🟢 ⚪️ ⚪️ ⚪️ - Only some basic Python knowledge is required (imports, lists, dict, loops, etc). 

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.1 Introduction to Content Chunks
</h3>

So far, all our user messages contained the simplest form of content: a string.

```python
    # Familiar syntax
    msg = UserMessage(content="How to reverse a Python list?")

    # Under the hood, this is equivalent to:
    msg = UserMessage(content=[{"type": "text", "text": "How to reverse a Python list?"}])
```
<br>

`content` can either contain a string or a list of **structured content chunks** of different **modalities** (text, audio, image, etc).


**Content** chunks can either be represented with a dict or a class-based notation. Very similar to what you already know for messages.

```python
    # dict notation
    {"type": "text", "text": "What is on this image?"}

    # class-based notation
    from mistralai.client.models import TextChunk
    TextChunk(text="What is on this image?")
```
<br>
The different possible types have been found in the source code (cf. contentchunk.py):<br><br>


  Class | Description |
 |----------|----------|
 | TextChunk(text) | `TextChunk(text="What is on this image?")`|
 | ImageURLChunk(image_url) | `image_url` can either be <br>- **A Web URL** (e.g. "https://upload.wikimedia.org/wikipedia/commons/d/de/Flag_of_the_United_States.png")<br>- **A base64 Encoded Image** (e.g. f"data:image/png;base64,{base64_image}")|
 | DocumentURLChunk(document_url) | `document_url` can either be:<br> - **A Web URL** (e.g. "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf") <br>- **A base64 Encoded Document** (e.g. f"data:application/pdf;base64,{base64_doc}")<br><br>For the most complicated use-cases, the optimized `ocr` endpoint is more appropriate.|
 | AudioChunk(input_audio) | A base64 encoded audio file (e.g. `AudioChunk(input_audio=base64_audio)`<br><br>For transcription, speeach generation and other advanced use-cases, the optimized `audio` endpoint is more appropriate.|
 | FileChunk(file_id) | For [Studio files](https://console.mistral.ai/build/files): they come from Document AI/OCR, Batches, Audio processing, etc and can also be uploaded via `mistral.files.upload()`|
 | ReferenceChunk(reference) | Citation of the document used for a response - Ignored for now but more info available in the [documentation](https://docs.mistral.ai/studio-api/conversations/citations)|
 | ThinkChunk(thinking) | Thinking traces when `reasoning` is set to `high` in `mistral.chat.complete()` - Ignored for now but more info available in the [documentation](https://docs.mistral.ai/studio-api/conversations/reasoning)|

---

In [15]:
import base64

def encode_file_to_base64(file_path: str) -> str:
    with open(file_path, "rb") as file:
        file_contents = file.read()
    return base64.b64encode(file_contents).decode("utf-8")

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.2 Working with Images
</h3>

In [16]:
from mistralai.client.models import UserMessage, TextChunk, ImageURLChunk
from IPython.display import Markdown

# Currently allowed formats: JPEG, PNG, WEBP, GIF, MPO, HEIF, AVIF, BMP, TIFF
img_url = r"https://upload.wikimedia.org/wikipedia/commons/d/de/Flag_of_the_United_States.png"

# Create a message with 2 modalities: text + image URL
msg = UserMessage(
    content=
    [
        TextChunk(text="What is on this image?"),
        ImageURLChunk(image_url=img_url)
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)


The image displays the flag of the United States of America. It consists of 13 horizontal stripes alternating in red and white, representing the original 13 colonies. In the canton (the upper left corner), there is a blue rectangle containing 50 white stars arranged in a 9x9 grid with alternating rows of five and six stars. The stars represent the 50 states of the United States.

In [17]:
# You can use the same technic to work with a local picture but need to convert it to base64
base64_image = encode_file_to_base64(r"images/free api.png")

msg = UserMessage(
    content=
    [
        TextChunk(text="What is on this image?"),
        ImageURLChunk(image_url=f"data:image/png;base64,{base64_image}")
    ]
)

response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

Markdown(response.choices[0].message.content)

The image appears to be a screenshot of a user interface related to subscriptions and API plans. Here are the details:

1. **Subscriptions Section:**
   - The title "Subscriptions" is prominently displayed at the top.
   - Two subscription options are shown:
     - **Vibe**: Represented by an orange icon with the letter "M".
     - **API Plan**: Represented by a blue icon with the letter "M".

2. **Free Mode Section:**
   - The title "Free mode" is displayed below the subscription options.
   - A description follows, stating that users can create API keys and use the free tier within the limits described on the limits page. This free usage is included in the Vibe subscription if available, or is available by default.

3. **Upgrade Button:**
   - On the right side of the "Free mode" section, there is a black button labeled "Upgrade".

The overall layout suggests a user interface for managing different subscription plans and understanding the available free tier usage.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.3 Working with Documents
</h3>

In [18]:
from mistralai.client.models import DocumentURLChunk

doc_url = r"https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"

# Create a message with 2 modalities: text + URL to a PDF file
msg = UserMessage(
    content=
    [
        TextChunk(text="What is this document about?"),
        DocumentURLChunk(document_url=doc_url)
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

This document introduces the **Transformer**, a novel neural network architecture for sequence transduction tasks (such as machine translation) that relies entirely on **attention mechanisms** instead of recurrent or convolutional layers. The key contributions and findings of the paper are:

### **Core Idea**
- The Transformer replaces recurrent neural networks (RNNs) and convolutional neural networks (CNNs) with **self-attention mechanisms**, enabling more efficient parallelization and better performance on long-range dependencies.
- It is the first model to achieve state-of-the-art results in machine translation **without using recurrence or convolutions**.

### **Key Innovations**
1. **Self-Attention Mechanism**
   - Uses **scaled dot-product attention** to compute relationships between all positions in a sequence simultaneously.
   - Introduces **multi-head attention**, allowing the model to jointly attend to information from different representation subspaces.

2. **Positional Encoding**
   - Adds **sinusoidal positional encodings** (or learned embeddings) to retain information about token order, since the model has no inherent notion of sequence.

3. **Encoder-Decoder Architecture**
   - The **encoder** consists of stacked self-attention and feed-forward layers.
   - The **decoder** includes an additional attention layer over the encoder’s output, with masking to prevent future token dependencies (ensuring autoregressive generation).

4. **Training Efficiency**
   - Achieves **faster training** (up to 3.5 days on 8 GPUs) compared to previous models (e.g., RNN-based or CNN-based architectures).
   - Uses **residual connections, layer normalization, and dropout** for stability and regularization.

### **Performance**
- **English-to-German Translation**: Achieves a **BLEU score of 28.4**, outperforming previous state-of-the-art models (including ensembles) by over 2 BLEU.
- **English-to-French Translation**: Achieves a **BLEU score of 41.0**, setting a new single-model benchmark.
- **Reduced Training Cost**: Requires significantly fewer computational resources than prior models.

### **Why Self-Attention?**
- **Parallelization**: Unlike RNNs, self-attention processes all positions in parallel, reducing training time.
- **Long-Range Dependencies**: Captures relationships between distant tokens more effectively than CNNs or RNNs.
- **Interpretability**: Attention weights provide insights into model decisions (e.g., syntactic/semantic alignment).

### **Future Work**
- Extending the Transformer to **other modalities** (e.g., images, audio).
- Exploring **local attention mechanisms** for efficiency with large inputs.

### **Impact**
The Transformer has become a foundational architecture in deep learning, influencing models like **BERT, GPT, and Vision Transformers (ViT)**. Its efficiency and scalability have made it a standard choice for NLP and beyond.

**Full paper**: [Attention Is All You Need (NIPS 2017)](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).

In [19]:
# You can use the same technic to work with a local document but need to convert it to base64
base64_doc = encode_file_to_base64(r"files/2509.18076v1.pdf")

msg = UserMessage(
    content=
    [
        TextChunk(text="What is this document about?"),
        DocumentURLChunk(document_url=f"data:application/pdf;base64,{base64_doc}")
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

The document presents a framework called **ToolGT** designed to improve the function-calling capabilities of large language models (LLMs) by incorporating structured reasoning templates. The key contributions include:

1. **Template-Based Reasoning**: A structured prompting method that guides LLMs through critical stages of function calling, such as tool identification, parameter extraction, and validation, to enhance accuracy and interpretability.

2. **Structured Reasoning Dataset (ToolGT)**: A synthetic dataset construction pipeline that generates high-quality reasoning chains and function calls, ensuring consistency and correctness. The dataset is validated through manual checks (Exact Match and AST) and LLM-based verification.

3. **Empirical Results**: Experiments on BFCLv2 and Nexus benchmarks show that both structured prompting and fine-tuning with ToolGT significantly improve function-calling performance across various models. Template-based approaches outperform free-form Chain-of-Thought (CoT) prompting and no-thought baselines, achieving up to 12% relative improvements.

4. **Limitations and Future Work**: The study acknowledges limitations such as dataset coverage gaps (e.g., nested function calls) and the need for adaptive template strategies. Future work includes expanding the dataset to cover more complex scenarios and developing models that dynamically select the most suitable reasoning template.

The document concludes by emphasizing the importance of structured reasoning for reliable and interpretable tool use in LLMs, laying the foundation for future advancements in LLM-based agents.

In [20]:
from mistralai.client.models import FileChunk
from IPython.display import display

# For Studio files (https://console.mistral.ai/build/files): they come from Document AI/OCR, Batches, Audio processing, etc and can also be uploaded via `mistral.files.upload()`
files = mistral.files.list().data

if len(files)>0:
    first_file_id=files[0].id

    msg = UserMessage(
        content=
        [
            FileChunk(file_id=first_file_id),
            TextChunk(text="What is this file about?")
        ]
    )

    response = mistral.chat.complete(
        model="mistral-small-latest",
        messages=[msg]
    )

    display(Markdown(response.choices[0].message.content))

else:
    print("No files available")

The document provides a guide on how to access and utilize the LangChain Documentation MCP Server to retrieve up-to-date LangChain documentation for use in an AI agent. It outlines a three-step process: selecting connectors, adding the MCP server, and enabling or disabling specific functions. The MCP server allows users to search LangChain's knowledge base, access documentation, and interact with a virtualized filesystem containing LangChain's documentation pages and OpenAPI specs. The document also includes a minimal example of how to create a Python agent in LangChain using the `create_agent` function, demonstrating the use of models, tools, and system prompts.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.4 Working with Audio
</h3>

In [21]:
from mistralai.client.models import AudioChunk

base64_audio = encode_file_to_base64(r"audio/boulangerie.m4a")

msg = UserMessage(
    content=
    [
        AudioChunk(input_audio=base64_audio),
        TextChunk(text="Transcribe this audio file into English please and tell me what language is spoken in the audio.")
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

**Transcription in English:**
*"Hello, so one baguette, two croissants, and a lemon tart, please. That’s all, thank you."*

**Language spoken in the audio:** French.